# Welcome to Week 5 - Agent Frameworks

## Day 3: Microsoft Agent Framework

Same week, same five steps, a new framework. The whole idea this week is that once you know one agent framework, you basically know them all, so every day we build the same agent the same five steps and watch the idioms rhyme:

1. **Create the agent** - give it a model and a system prompt.
2. **Run it** - send a message, get a reply.
3. **Add tools** - plain typed functions the agent can call.
4. **Add MCP** - connect a tool server someone else wrote, wired the same way every time.
5. **Put it in a loop with a goal** - hand it an objective and let it work, step after step, until the job is done.

Steps 1 and 2 are still just an LLM call. Tools and MCP give the agent things it can do. Step 5 is where it becomes an agent: the framework runs the loop on its own, choosing a tool, reading the result, choosing again, until the goal is met.

The running project is the same SQLite todo board from Day 1. A worker picks one goal off the board, plans its own steps, does the work with its agent loop, and ticks each step off. The board code (`board.py`) is the same file, byte for byte; only the framework around it changes.

Today is the **Microsoft Agent Framework**, Microsoft's official agent SDK. It has a backstory worth knowing: Microsoft ran two separate agent efforts for years, AutoGen for multi-agent research and Semantic Kernel for production plumbing, and this framework is the two converging into one. You get a single `Agent` that looks almost the same in Python and C#, and underneath it carries a graph-based workflow engine for durable, long-running processes. We use the plain agent here and look at the workflow engine at the end.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The Microsoft Agent Framework docs</h2>
            <span style="color:#00bfff;">The docs live at <a href="https://learn.microsoft.com/en-us/agent-framework/">learn.microsoft.com/agent-framework</a>. The framework reached 1.0 in April 2026 and renamed a lot of its API on the way, so we pin the version (1.8.1 here) and treat older blog posts with care. Two shapes to know: the OpenAI client takes the model as <code>model=</code>, and an <code>Agent</code> is built from a <code>client</code> plus <code>instructions</code>.</span>
        </td>
    </tr>
</table>

## Setup

Two things this day needs, both already in place from earlier weeks:

- **Node**, for `npx` (the filesystem MCP server runs over it). Check with `node --version`.
- **`OPENAI_API_KEY`** in the repo-root `.env`. Today's frameworks use OpenAI's `gpt-5.4-mini`, and that key has been in your `.env` since Week 1.

The Microsoft Agent Framework is part of the repo's environment, so a normal `uv sync` from the repo root installs everything. Open this notebook in Cursor and pick the repo's default **Python 3.12.12** kernel, the same one you use every week, then run the cells top to bottom.

Warm the filesystem MCP server once now so the first run is fast, then stop it with Ctrl-C as soon as it prints that it is running:

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [1]:
# The framework prints a couple of "experimental" notices on import; quiet them.
import warnings
warnings.filterwarnings("ignore", message=r".*experimental.*")

In [2]:

import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient

load_dotenv(override=True)


True

## Step 1: Create the agent

In the Microsoft Agent Framework an agent is an `Agent`: a `client` for the model plus `instructions`, which is its system prompt. We build the client once as an `OpenAIChatClient` pointed at `gpt-5.4-mini`, reading `OPENAI_API_KEY` from the environment, and reuse it everywhere. The framework is async-first, so we always `await` the agent.

In [3]:
# 사용할 모델명
MODEL = "gpt-5.6-luna"

# 지정한 모델로 OpenAI 채팅 클라이언트를 생성한다.
client = OpenAIChatClient(model=MODEL)

# 지시문을 지정해 에이전트를 생성한다.
agent = Agent(
    client=client,
    instructions="너는 간결하고 친근한 어시스턴트다. 짧은 한 문장으로 답하라.",
)

## Step 2: Run it

Send a message, await the reply, and print the result's `.text`. With no tools yet there is nothing to loop over, so the agent just answers. This is still only an LLM call.

In [4]:
# 에이전트에 입력 메시지를 전달해 실행하고 결과를 받는다.
result = await agent.run("스페인어로 인사말을 말해 줘.")

# 결과 텍스트를 출력한다.
print(result.text)

¡Hola! 만나서 반가워요.


## Our project this week: a SQLite todo board

The worker coordinates through the same tiny SQLite board as Day 1, the same `board.py` file: one file, one table, no server to run. A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end. Under the hood the board is just a list of dicts (a goal has `parent_id` of None, a step points at its goal):

In [5]:
import board

# 보드를 초기화해 기존 todo 항목을 모두 비운다.
board.reset_board()

# 목표를 하나 추가한다.
board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.")

# 보드에 있는 모든 todo 항목을 조회한다.
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [6]:
board.show_board()

Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.

## Step 3: Add tools

A tool in the Microsoft Agent Framework is a plain typed Python function. The framework reads the type hints and the docstring and builds the JSON schema for you, so there is nothing else to declare; the `@tool` decorator exists only when you want to rename or constrain something. You hand the functions to the agent in its `tools=[...]` list.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [7]:
def show_todos() -> list[dict]:
    """보드에 있는 모든 todo 항목을 나열한다. 목표(goal)는 parent_id가 None이고, 단계(step)는 parent_id에 소속 목표의 id가 들어 있다."""
    # 보드 모듈에서 전체 todo 목록을 가져와 그대로 반환한다.
    return board.list_todos()


def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """하나의 목표를 순서가 있는 단계 체크리스트로 나누어 보드에 등록한다. 목표의 id와 짧은 단계 설명 목록을 전달한다."""
    # 전달받은 각 단계 설명을 해당 목표의 하위 단계로 추가하고, 생성된 단계 id 목록을 반환한다.
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


def complete_task(task_id: int, result: str) -> dict:
    """해당 id의 todo(단계 또는 목표)를 완료 상태로 표시하고 짧은 결과 요약을 기록한다."""
    # 보드에서 해당 항목을 완료 처리한다.
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [8]:
# 지시문과 도구 목록을 지정해 보드 관리용 에이전트를 생성한다.
board_agent = Agent(
    client=client,
    instructions="너는 공유 todo 보드를 관리하는 것을 돕는다.",
    tools=[show_todos, complete_task],
)

In [9]:
# 에이전트에 입력 메시지를 전달해 실행하고 결과를 받는다.
result = await board_agent.run("지금 보드에 무엇이 올라와 있고, 각각의 상태는 어떤가?")

# 결과 텍스트를 출력한다.
print(result.text)

현재 보드에는 다음 항목이 올라와 있습니다.

1. **notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤 spanish.txt에 작성하기**
   - 상태: **대기 중**
   - 결과: 아직 기록되지 않음


## Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server, the same Node server every framework uses this week, scoped to a single `workspace` folder so it can only touch files in there. In the Microsoft Agent Framework an MCP server is an `MCPStdioTool`: you open it with `async with` and pass it to the agent in the same `tools=[...]` list.

The framework's `MCPStdioTool` does not expose the server's stderr, so we subclass it to set two things on the underlying stdio client. `errlog=subprocess.DEVNULL` quiets the server's startup banner and lets it run from a Jupyter kernel on Windows, where the kernel's stderr has no real file descriptor. `cwd` starts the server in the workspace, so the agent's file names resolve there. On Mac and Linux the errlog simply keeps the output clean.

In [10]:
# 에이전트가 접근할 수 있는 유일한 폴더의 절대 경로
workspace = Path("workspace").resolve()


class FilesystemMCP(MCPStdioTool):
    """stderr를 DEVNULL로 보내고 작업 디렉터리를 workspace로 지정한 파일시스템 서버.
    이렇게 하면 파일명이 해당 폴더 기준으로 해석되고, Windows의 Jupyter 커널에서
    출력 오염 없이 실행된다."""

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client

        # 서버 실행 파라미터를 구성하며, cwd를 workspace로 지정한다.
        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        # stderr 출력을 버리도록 errlog를 DEVNULL로 지정해 stdio 클라이언트를 만든다.
        return stdio_client(server=params, errlog=subprocess.DEVNULL)


# 파일시스템 MCP 서버를 stdio 방식으로 실행하는 도구를 생성한다.
filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
)

In [11]:
# 지시문과 파일시스템 도구를 지정해 파일 작업용 에이전트를 생성한다.
file_agent = Agent(
    client=client,
    instructions="너는 네 작업 폴더 안의 파일을 읽고 쓸 수 있다. 요청받은 일을 네 도구로 수행하라.",
    tools=[filesystem],
)

In [12]:
# with 블록 안에서 MCP 서버 프로세스가 실행되고, 블록을 벗어나면 정리된다.
async with filesystem:
    # 에이전트에 입력 메시지를 전달해 실행하고 결과를 받는다.
    result = await file_agent.run("notes.txt를 읽고 짧은 한 문장으로 요약해 줘.")

# 결과 텍스트를 출력한다.
print(result.text)

팀원들이 공동 작업 게시판의 과제를 하나씩 맡아 신중히 수행하고 완료 처리하며 작은 언어 튜터를 만들어 간다는 내용입니다.


## Step 5: Put it in a loop with a goal

Now the payoff. Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat. Watch the board fill with steps and then strike them through.

In [13]:
# 에이전트에게 주어지는 시스템 지시문
INSTRUCTIONS = """
너는 공유 todo 보드와 여러 파일 도구를 다루는 신중한 작업자다.

대기 중인 목표를 맡아 끝까지 완수하라. 먼저 짧은 계획을 세우는 것부터 시작하라. 즉, 그 작업이 나뉘는 구체적인 몇 개의 단계를 보드의 해당 목표 아래에 추가하라. 그런 다음 파일 도구로 그 단계들을 수행하고, 각 단계를 끝낼 때마다 완료로 표시하라. 모든 단계가 끝나면 목표를 닫아라. 네 파일들은 도구가 사용할 수 있도록 허용된 단 하나의 폴더 안에 있다.
"""

# 지시문과 도구 목록을 지정해 워커 에이전트를 생성한다.
worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

# 보드를 초기화해 기존 todo 항목을 모두 비운다.
board.reset_board()

# 목표를 하나 추가하고 그 id를 받는다.
goal_id = board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.")

# 워커가 해당 목표를 집어 든다: pending -> in_progress 상태로 변경된다.
board.claim_todo(goal_id)

# with 블록 안에서 MCP 서버 프로세스가 실행되고, 블록을 벗어나면 정리된다.
async with filesystem:
    # 에이전트에 입력 메시지를 전달해 실행한다.
    await worker.run("보드에 대기 중인 목표를 처리해 줘.")

# 실행 후 보드 상태를 출력한다.
board.show_board()

Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.  notes.txt를 자연스러운 스페인어로 번역해 spanish.txt에 작성하고 검토까지 완료했다.
  Step #2: 작업 폴더와 notes.txt의 내용을 확인한다.  작업 폴더를 확인하고 notes.txt의 전체 내용을 읽었다.
  Step #3: notes.txt 내용을 자연스러운 스페인어로 번역한다.  notes.txt의 안내문을 자연스러운 스페인어로 번역했다.
  Step #4: 번역문을 spanish.txt에 작성하고 결과를 검토한다.  번역문을 spanish.txt에 작성하고 파일 내용을 다시 읽어 정확성을 확인했다.

## Run the same worker from the terminal

Everything you just watched in step 5 is also packaged as a small script, `maf_worker.py`, sitting next to this notebook. It seeds the same goal, builds the same agent with the same three board tools and the filesystem MCP server, and runs the same loop, only from the command line instead of the kernel. Open a terminal in this folder and run it:

```bash
uv run maf_worker.py
```

You will see the agent plan its steps, work the goal and tick each one off, then the finished board and the Spanish it wrote. This is the shape every worker takes on Day 5, where a Google ADK orchestrator launches one of these per framework as a parallel subprocess against a single shared board.

## The one cool thing: durable workflows, and one API in two languages

The plain `Agent` is the easy surface. The reason enterprises take this framework seriously is the layer underneath: a graph-based **workflow engine**. You wire agents and plain functions together as typed nodes and edges, and the framework runs them as a durable, restartable process with checkpointing, streaming and human-in-the-loop approval. It is the concrete payoff of the AutoGen and Semantic Kernel merge, and it is what you reach for when a job runs long enough that it has to survive a restart. Roughly the shape, not something we run here:

```python
from agent_framework import WorkflowBuilder

workflow = (
    WorkflowBuilder()
    .add_edge(researcher, writer)     # each node is an agent or a plain function
    .add_edge(writer, reviewer)
    .build()
)
```

The same `Agent` and `OpenAIChatClient` shapes exist in C#, so a team can move a design between Python and .NET almost line for line. And the same one-line spirit as the rest of the week: `OpenAIChatClient` takes a `base_url`, so to run on any OpenAI-compatible endpoint you point it there and change nothing else:

```python
client = OpenAIChatClient(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY, model="openai/gpt-5.4-mini")
```

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Seed a different goal on the board, for example "write a short haiku about Madrid into madrid.txt", and run the worker again. Does it plan sensible steps and pick the right file tools? Then point the <code>OpenAIChatClient</code> at another OpenAI-compatible endpoint you have access to by passing a <code>base_url</code>, rerun the worker, and watch the same agent run on a different model.</span>
        </td>
    </tr>
</table>